# Train a "products in a room" LoRA on Stable Diffusion 3.5 Medium (Google Colab)

**How this compares to the other notebooks in this folder:**

| | SD1.5 | SDXL | Stable Diffusion 3.5 Medium | Qwen-Image |
|---|---|---|---|---|
| Base model size | ~0.9B | ~2.6B | ~2.5B | ~20B |
| Objective | DDPM noise-prediction | DDPM noise-prediction | flow matching | flow matching |
| Minimum realistic GPU | T4 (free) | T4 (free, tight) | L4 GPU | A100 40GB (Pro+) |
| Training recipe | hand-written loop | hand-written loop | HF's maintained script via `accelerate` | HF's maintained script via `accelerate` |

**Why this one uses HF's own script instead of a hand-written loop (same reasoning as the Qwen-Image notebook).**
Stable Diffusion 3.5 Medium is a flow-matching transformer, not a DDPM UNet like SD1.5/SDXL — a meaningfully different, newer training
recipe (sample a continuous noise level, predict a velocity field, rather than predict discrete-timestep noise).
Rather than this notebook's author hand-rolling that forward/backward pass and risking a subtly wrong-but-plausible
implementation, Section 6 drives Hugging Face's own maintained LoRA training script for Stable Diffusion 3.5 Medium via
`accelerate launch`, the same pattern as the Qwen-Image notebook. Sections 1-5 (finding, validating, captioning,
compositing the images, and writing the dataset to disk) are shared with that approach too.

## How to run it
1. Colab: **Runtime -> Change runtime type -> L4 GPU.**
2. Put your image folder in Google Drive, same structure as the other notebooks.
3. Leave `DRIVE_IMAGES_DIR = "auto"` in the **CONFIG** cell.
4. **Run all.** Section 6 checks your GPU (needs ~18GB+) before continuing.

| Steps | Estimated time (rough, unmeasured - Section 7 measures your real rate) |
|---|---|
| 500 | 35-70 min |
| 1000 (default) | 70-140 min |

5. The trained LoRA is written to `DRIVE_OUTPUT_DIR`, checkpointed every `SAVE_EVERY` steps.


In [ ]:
# ============================== CONFIG: edit these ==============================
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/Curalina - Humber Capstone/Supplier Images"   # "auto" = search your Drive + shared drives for the supplier folders.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/curalina_sd35_lora_output"

LOCAL_IMAGES_DIR = "/Users/rjsalmon/Documents/Humber/misc.curalina/Supplier Images"
LOCAL_OUTPUT_DIR = "./sd35_lora_output"

BASE_MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
TRIGGER = "crlnstyle"
RESOLUTION = 768
MAX_TRAIN_STEPS = 1000
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
LORA_RANK = 16
SAVE_EVERY = 250
GRADIENT_CHECKPOINTING = True
QUANTIZE_4BIT = True  # SD3.5 Medium's 2.5B transformer technically fits unquantized on an L4 (24GB); quantizing
# anyway keeps headroom for the T5 text encoder (~4.7B) running alongside it.
P_ROOM_BACKGROUND = 0.8
MIN_SIDE_PX = 200
SEED = 42

# Debug: set to a small number (e.g. 20) to try the pipeline on a few images first, or None for everything.
LIMIT_IMAGES = None


In [ ]:
import os, sys, json, random, math, shutil, re, time, difflib
from pathlib import Path
from collections import Counter

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def explain_missing(path: Path) -> str:
    # Finds the deepest folder that DOES exist, lists what is in it, and suggests the closest spelling.
    p = Path(path)
    missing = []
    while not p.exists() and p != p.parent:
        missing.insert(0, p.name)
        p = p.parent
    try:
        children = sorted(os.listdir(p))
    except OSError:
        children = []
    msg = f"'{path}' was not found.\nThe part that exists is: {p}\nThe first name that is missing is: '{missing[0] if missing else ''}'\n"
    msg += f"Folders that DO exist inside {p}:\n  " + ("\n  ".join(children[:40]) if children else "(none - is this the Google account that has access?)") + "\n"
    if missing and children:
        close = difflib.get_close_matches(missing[0], children, n=3, cutoff=0.4)
        if close:
            msg += f"Did you mean: {close}\n"
    msg += "Tip: in Colab open the Files panel (folder icon, left), browse to the folder, right-click -> Copy path, and paste it into the CONFIG cell."
    return msg


SUPPLIER_MARKERS = {"atriani", "celadon", "lazzoni", "luxus"}


def find_images_root(max_depth: int = 5):
    # Looks through every shared drive and My Drive for a folder that contains at least two of the supplier folders.
    roots = []
    shared = Path("/content/drive/Shareddrives")
    if shared.is_dir():
        roots += [shared / d for d in sorted(os.listdir(shared))]
    roots.append(Path("/content/drive/MyDrive"))
    hits = []
    for base in roots:
        base_parts = len(base.parts)
        for cur, dirs, _ in os.walk(base):
            depth = len(Path(cur).parts) - base_parts
            dirs[:] = [d for d in dirs if not d.startswith(".")] if depth < max_depth else []
            if len({d.lower() for d in dirs} & SUPPLIER_MARKERS) >= 2:
                hits.append(Path(cur))
                dirs[:] = []                       # found it; do not search inside
    return hits


if IN_COLAB:
    # Uncomment if a package is missing. Colab already has torch/torchvision.
    # !pip -q install diffusers transformers accelerate peft
    from google.colab import drive
    drive.mount("/content/drive",force_remount=True)
    for root in ["/content/drive", "/content/drive/Shareddrives", "/content/drive/MyDrive"]:
        print(f"{root} ->", sorted(os.listdir(root))[:30] if os.path.isdir(root) else "(not found)")

    # Make sure to run the CONFIG cell (cell b568965d) first to set DRIVE_IMAGES_DIR correctly!
    print(f"DEBUG (Cell 6f9d2616): DRIVE_IMAGES_DIR is '{DRIVE_IMAGES_DIR}'")

    if DRIVE_IMAGES_DIR.strip().lower() == "auto":
        print("Searching your Drive for the supplier image folders (looks for LUXUS / CELADON / ATRIANI / LAZZONI)...")
        hits = find_images_root()
        if len(hits) == 1:
            IMAGES_DIR = hits[0]
            print("Found:", IMAGES_DIR)
        elif len(hits) > 1:
            raise SystemExit("Found several candidate folders - copy one into DRIVE_IMAGES_DIR:\n  " + "\n  ".join(map(str, hits)))
        else:
            raise SystemExit("Could not find a folder containing LUXUS / CELADON / ATRIANI / LAZZONI anywhere in your Drive.\n"
                             "Check you mounted the Google account that has access, and that the shared drive is listed above.")
    else:
        IMAGES_DIR = Path(DRIVE_IMAGES_DIR)
    OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
    WORK_DIR = Path("/content/work")

else:
    IMAGES_DIR = Path(LOCAL_IMAGES_DIR)
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)
    WORK_DIR = Path("./work")

# Check the input folder FIRST, with a readable message, before creating anything.
print(f"DEBUG (Cell 6f9d2616): IMAGES_DIR is '{IMAGES_DIR}' before checking existence.")
if not IMAGES_DIR.is_dir():
    raise SystemExit("IMAGE FOLDER PROBLEM\n" + explain_missing(IMAGES_DIR))

*(Identical to the SD1.5 notebook from here through Section 4 - same data, same logic.)*

## 1. Find the images (and skip everything else)

We walk the whole folder tree. Only these extensions are treated as images; anything else — the Lazzoni PDFs,
`.DS_Store`, spreadsheets — is counted and skipped. Then every candidate is opened once with Pillow: files that are
corrupt, unreadable or smaller than `MIN_SIDE_PX` are dropped and listed, so training never hits a bad file.


In [ ]:
import pandas as pd
from PIL import Image, UnidentifiedImageError

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
Image.MAX_IMAGE_PIXELS = 60_000_000  # guard against decompression bombs

found, skipped_non_image = [], Counter()
for root, dirs, files in os.walk(IMAGES_DIR):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    for name in files:
        if name.startswith("."):
            continue
        p = Path(root) / name
        ext = p.suffix.lower()
        if ext in IMAGE_EXTS:
            found.append(p)
        else:
            skipped_non_image[ext or "(no extension)"] += 1

print(f"{len(found)} candidate image files")
print("skipped non-image files by type:", dict(skipped_non_image) or "none")

valid, rejected = [], []
for p in sorted(found):
    try:
        with Image.open(p) as im:
            im.verify()                      # cheap integrity check
        with Image.open(p) as im:            # verify() invalidates the handle, reopen for size
            w, h = im.size
        if min(w, h) < MIN_SIDE_PX:
            rejected.append((str(p.relative_to(IMAGES_DIR)), f"too small ({w}x{h})"))
        else:
            valid.append(p)
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as e:
        rejected.append((str(p.relative_to(IMAGES_DIR)), f"unreadable: {type(e).__name__}"))

if LIMIT_IMAGES:
    random.Random(SEED).shuffle(valid)
    valid = sorted(valid[:LIMIT_IMAGES])

print(f"{len(valid)} usable images, {len(rejected)} rejected")
pd.DataFrame(rejected, columns=["file", "reason"]).head(15)


## 2. Captions from the folder structure

Supplier = the top-level folder (`LUXUS`, `CELADON`, ...). Product name = the folder the image sits in, with the
`001 - ` numbering and any trailing SKU removed. A keyword pass maps names to a furniture category and a sensible room
(a bed goes in a bedroom, a dining chair in a dining room). Nothing here needs the spreadsheets.


In [ ]:
CATEGORY_RULES = [  # first match wins; (category, regex, rooms it belongs in)
    ("dining chair", r"dining chair", ["dining room"]),
    ("dining table", r"dining table", ["dining room"]),
    ("accent chair", r"accent chair|armchair|lounge chair|poltrona|chair", ["living room", "bedroom"]),
    ("coffee table", r"coffee table|cocktail table", ["living room"]),
    ("side table", r"side table|end table", ["living room", "bedroom"]),
    ("bar cabinet", r"bar cabinet", ["dining room", "living room"]),
    ("sectional sofa", r"sectional|modular", ["living room"]),
    ("sofa", r"\bsofa\b|\bloveseat\b", ["living room"]),
    ("ottoman", r"ottoman", ["living room"]),
    ("bench", r"\bbench\b", ["bedroom", "living room"]),
    ("nightstand", r"night ?stand|bedside", ["bedroom"]),
    ("bed", r"\bbed\b|\bking\b|\bqueen\b", ["bedroom"]),
    ("sideboard", r"sideboard|buffet|credenza", ["dining room", "living room"]),
    ("table", r"table", ["dining room", "living room"]),
    ("console or cabinet", r"console|media|cabinet|shelf|bookcase|vanity|desk|stool", ["living room", "dining room"]),
]


def clean_name(folder_name: str) -> str:
    n = re.sub(r"^\s*\d+\s*-\s*", "", folder_name)          # "001 - Halo Nightstand" -> "Halo Nightstand"
    n = re.sub(r"\s*-\s*\d{4,}\s*$", "", n)                # Celadon "Traverse I - 21368" -> "Traverse I"
    n = n.replace("_", ":").strip()
    return n or folder_name


def describe(path: Path):
    rel = path.relative_to(IMAGES_DIR)
    parts = rel.parts
    supplier = parts[0] if len(parts) > 1 else "unknown"
    folder = parts[-2] if len(parts) > 2 else path.stem        # product folder; fall back to file name
    if folder.lower() in {"product images", supplier.lower()}:
        folder = path.stem
    name = clean_name(folder)
    if supplier.upper() == "CELADON":
        return dict(supplier=supplier, name=name, category="framed wall art", rooms=["living room", "bedroom"], is_art=True)
    for category, pattern, rooms in CATEGORY_RULES:
        if re.search(pattern, name.lower()):
            return dict(supplier=supplier, name=name, category=category, rooms=rooms, is_art=False)
    return dict(supplier=supplier, name=name, category="piece of furniture", rooms=["living room"], is_art=False)


records = []
for p in valid:
    d = describe(p)
    d["path"] = str(p)
    records.append(d)

df = pd.DataFrame(records)
print(df.groupby(["supplier", "category"]).size().to_string())
df[["supplier", "name", "category"]].sample(min(8, len(df)), random_state=SEED)


## 3. Cut out the products and build "product in a room" training images

Studio photos have a white background, which would teach the model "furniture floats on white". So we **remove the
background** (flood-fill from the image border — white areas *inside* the product, like a white sofa, are kept) and paste
the cutout into a randomly generated room: random wall colour, floor, baseboard, sometimes a window, with a soft shadow.
Artwork is hung on the wall instead of placed on the floor. Cutouts are cached in `WORK_DIR`.


In [ ]:
import numpy as np
from PIL import ImageDraw, ImageFilter

CUTOUT_DIR = WORK_DIR / "cutouts"
CUTOUT_DIR.mkdir(parents=True, exist_ok=True)


def matte_white_background(img: Image.Image, tolerance: int = 45, max_side: int = 640) -> Image.Image:
    rgba = img.convert("RGBA")
    if max(rgba.size) > max_side:
        rgba.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    flat = Image.alpha_composite(Image.new("RGBA", rgba.size, (255, 255, 255, 255)), rgba).convert("RGB")
    work = flat.copy()
    w, h = work.size
    marker = (255, 0, 255)
    floor = 255 - tolerance
    for seed in [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1), (w // 2, 0), (w // 2, h - 1), (0, h // 2), (w - 1, h // 2)]:
        px = flat.getpixel(seed)
        if all(int(c) >= floor for c in px[:3]):
            ImageDraw.floodfill(work, seed, marker, thresh=tolerance)
    arr = np.array(rgba)
    background = np.all(np.array(work) == np.array(marker), axis=2)
    arr[..., 3][background] = 0
    arr[..., 3][arr[..., 3] < 40] = 0
    alpha = Image.fromarray(arr[..., 3]).filter(ImageFilter.MinFilter(3))   # erode 1px: drops the white fringe
    arr[..., 3] = np.array(alpha)
    return Image.fromarray(arr, "RGBA")


SKIP_REASONS = Counter()
SWATCH_NAME = re.compile(r"^(l\d+|ai\d+)\b", re.I)   # finish/material swatch files like "L139.png", "AI01- Chrome.webp"


def build_cutout(path: str, is_art: bool):
    # Returns the cached cutout path, or None (and counts why) if the image is not a usable product photo.
    p = Path(path)
    if SWATCH_NAME.match(p.stem):
        SKIP_REASONS["finish swatch (by file name)"] += 1
        return None
    out = CUTOUT_DIR / (re.sub(r"[^A-Za-z0-9]+", "_", str(p.relative_to(IMAGES_DIR))) + ".png")
    if out.exists():
        return str(out)
    try:
        with Image.open(p) as im:
            cut = matte_white_background(im)
        alpha = np.array(cut)[..., 3]
        visible = (alpha > 127).mean()
        if visible < 0.02:
            SKIP_REASONS["almost all background"] += 1
            return None
        if not is_art and visible > 0.93:
            SKIP_REASONS["no studio background (swatch / close-up)"] += 1
            return None
        rgb = np.array(cut.convert("RGB")).astype(float)
        if not is_art and rgb[alpha > 127].mean() > 240:      # mostly white pixels: a line drawing, not a photo
            SKIP_REASONS["line drawing (white on white)"] += 1
            return None
        cut.crop(cut.getchannel("A").getbbox()).save(out)
        return str(out)
    except Exception as e:                        # never let one bad file stop the run
        SKIP_REASONS[f"error: {type(e).__name__}"] += 1
        return None


t0 = time.time()
df["cutout"] = [build_cutout(p, a) for p, a in zip(df["path"], df["is_art"])]
before = len(df)
df = df[df["cutout"].notna()].reset_index(drop=True)      # only clean product cutouts are trained on
print(f"{len(df)}/{before} images kept after cutout ({time.time() - t0:.0f}s). Dropped: {dict(SKIP_REASONS) or 'none'}")

WALLS = [(236, 231, 222), (222, 214, 200), (205, 210, 205), (196, 187, 175), (233, 224, 214),
         (94, 98, 96), (58, 56, 60), (170, 150, 128), (215, 220, 224)]
FLOORS = [(176, 140, 100), (140, 105, 75), (205, 190, 165), (95, 80, 70), (160, 150, 140), (60, 55, 52)]


def make_room_background(rng: random.Random, size: int = 512):
    wall, floor = rng.choice(WALLS), rng.choice(FLOORS)
    horizon = int(size * rng.uniform(0.58, 0.68))
    img = Image.new("RGB", (size, size), wall)
    d = ImageDraw.Draw(img)
    d.rectangle([0, horizon, size, size], fill=floor)
    d.rectangle([0, horizon - 10, size, horizon], fill=tuple(min(255, c + 25) for c in wall))   # baseboard
    if rng.random() < 0.5:                                                                      # window
        x0 = rng.randint(int(size * 0.08), int(size * 0.55)); w = rng.randint(90, 150)
        d.rectangle([x0, int(size * 0.12), x0 + w, horizon - 60], fill=(250, 250, 244), outline=(120, 120, 110), width=4)
    grad = np.linspace(1.06, 0.92, size)[:, None, None]
    return Image.fromarray(np.clip(np.array(img) * grad, 0, 255).astype("uint8")), horizon


def compose_room_sample(cut: Image.Image, is_art: bool, rng: random.Random, size: int = 512):
    bg, horizon = make_room_background(rng, size)
    cw, ch = cut.size
    if is_art:
        scale = rng.uniform(0.28, 0.5) * size / max(cw, ch)
    else:
        scale = rng.uniform(0.45, 0.85) * size / max(cw, ch)
    cut = cut.resize((max(1, int(cw * scale)), max(1, int(ch * scale))), Image.Resampling.LANCZOS)
    cw, ch = cut.size
    x = int(size / 2 - cw / 2 + rng.uniform(-0.12, 0.12) * size)
    if is_art:
        y = int(horizon * rng.uniform(0.12, 0.35))
    else:
        y = min(size - ch, int(horizon + (size - horizon) * rng.uniform(0.15, 0.55)) - ch)
        shadow = Image.new("RGBA", bg.size, (0, 0, 0, 0))
        ImageDraw.Draw(shadow).ellipse([x, y + ch - int(ch * 0.08), x + cw, y + ch + int(ch * 0.1)], fill=(0, 0, 0, 90))
        bg = Image.alpha_composite(bg.convert("RGBA"), shadow.filter(ImageFilter.GaussianBlur(9))).convert("RGB")
    bg.paste(cut, (x, max(0, y)), cut)
    return bg


def plain_product_sample(cutout_path: str, size: int = 512):
    with Image.open(cutout_path) as im:
        im = im.convert("RGBA")
        flat = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im).convert("RGB")
    flat.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (size, size), (255, 255, 255))
    canvas.paste(flat, ((size - flat.width) // 2, (size - flat.height) // 2))
    return canvas


def make_sample(row, rng: random.Random, size: int = 512):
    """Returns (PIL image, caption): a product pasted into a synthetic room, or a plain white-background product photo."""
    label = f"{row['name']}" if row["category"] in row["name"].lower() else f"{row['name']} {row['category']}"
    if rng.random() < P_ROOM_BACKGROUND:
        room = rng.choice(row["rooms"])
        with Image.open(row["cutout"]) as c:
            image = compose_room_sample(c.convert("RGBA"), row["is_art"], rng, size)
        caption = f"{TRIGGER}, a {label} in a {room}, interior design photo"
    else:
        image = plain_product_sample(row["cutout"], size)
        caption = f"{TRIGGER}, product photo of a {label}, white background"
    return image, caption


In [ ]:
import matplotlib.pyplot as plt

rng_preview = random.Random(SEED)
sample_rows = df.sample(min(8, len(df)), random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, (_, row) in zip(axes.ravel(), sample_rows.iterrows()):
    img, cap = make_sample(row, rng_preview)
    ax.imshow(img); ax.axis("off"); ax.set_title(cap[:48] + "...", fontsize=8)
fig.suptitle("Training samples (background-removed products pasted into synthetic rooms) - same data as the SD1.5 run")
fig.tight_layout(); plt.show()


## 5. Write the dataset the way the training script expects it

Same reasoning as the Qwen-Image notebook: materializes `metadata.jsonl` + images to disk once, up front, instead of generating fresh compositions inside a hand-written loop.

In [ ]:
DATASET_DIR = WORK_DIR / "train_dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = max(len(df), min(4000, MAX_TRAIN_STEPS * BATCH_SIZE * 2))
rng_ds = random.Random(SEED)

metadata_path = DATASET_DIR / "metadata.jsonl"
written = 0
with open(metadata_path, "w") as f:
    for i in range(N_SAMPLES):
        row = df.iloc[i % len(df)]
        try:
            image, caption = make_sample(row, rng_ds, RESOLUTION)
        except Exception as e:
            print(f"skipping sample {i} ({row['name']}): {type(e).__name__}: {e}")
            continue
        fname = f"sample_{i:05d}.jpg"
        image.convert("RGB").save(DATASET_DIR / fname, quality=92)
        f.write(json.dumps({"file_name": fname, "text": caption}) + "\n")
        written += 1

print(f"wrote {written} training images + captions to {DATASET_DIR}")


## 6. GPU check

Stable Diffusion 3.5 Medium needs roughly 18GB+ VRAM even quantized. This checks before you spend time downloading weights.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU found. In Colab: Runtime > Change runtime type > L4 GPU, then run all again.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name}  ({total_vram_gb:.0f} GB)")

MIN_VRAM_GB = 18
if total_vram_gb < MIN_VRAM_GB:
    raise SystemExit(
        f"This GPU reports {total_vram_gb:.0f} GB. Stable Diffusion 3.5 Medium needs roughly {MIN_VRAM_GB}GB+ even with "
        f"quantization. In Colab: Runtime > Change runtime type > L4 GPU. This check exists so you find out "
        f"now, not after a long download and a partial run."
    )
print("GPU check passed - continuing.")


## 7. Install `diffusers` from source and locate the Stable Diffusion 3.5 Medium LoRA training script

Searches rather than assumes one exact script path, and fails with a clear message (not a wrong guess) if the name has moved in the `diffusers` version you end up with.

In [ ]:
import subprocess, glob

!pip install -q -U git+https://github.com/huggingface/diffusers
!pip install -q -U transformers accelerate peft bitsandbytes sentencepiece protobuf

if not Path("diffusers").is_dir():
    !git clone -q --depth 1 https://github.com/huggingface/diffusers.git

candidates = sorted(set(sum([glob.glob(p) for p in ['diffusers/examples/dreambooth/train_dreambooth_lora_sd3.py', 'diffusers/examples/*/*sd3*lora*.py']], [])))
print("candidate training scripts found:", candidates or "(none)")

if not candidates:
    available = sorted(glob.glob("diffusers/examples/*/train_*lora*.py"))
    raise SystemExit(
        "Could not find a Stable Diffusion 3.5 Medium LoRA training script in this diffusers checkout under examples/. This is a "
        "real 'needs_input', not something to guess past.\n\nOther LoRA training scripts that DO exist (for "
        "reference, to see the current naming pattern):\n  " + "\n  ".join(available[:20]) + "\n\n"
        "Check https://github.com/huggingface/diffusers/tree/main/examples for the current script name and "
        "update TRAIN_SCRIPT below by hand."
    )

TRAIN_SCRIPT = candidates[0]
print("using:", TRAIN_SCRIPT)
!python {TRAIN_SCRIPT} --help 2>&1 | head -5


## 8. Train

Streams the script's own output live, same pattern as the Qwen-Image notebook.

In [ ]:
import subprocess, time

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_output_dir = OUTPUT_DIR / "final"

cmd = [
    "accelerate", "launch", TRAIN_SCRIPT,
    f"--pretrained_model_name_or_path={BASE_MODEL_ID}",
    f"--instance_data_dir={DATASET_DIR}",
    f"--output_dir={run_output_dir}",
    f"--instance_prompt={TRIGGER}",
    f"--resolution={RESOLUTION}",
    f"--train_batch_size={BATCH_SIZE}",
    f"--max_train_steps={MAX_TRAIN_STEPS}",
    f"--learning_rate={LEARNING_RATE}",
    f"--rank={LORA_RANK}",
    f"--checkpointing_steps={SAVE_EVERY}",
    f"--seed={SEED}",
    "--mixed_precision=bf16",
    "--use_8bit_adam",
]
if GRADIENT_CHECKPOINTING:
    cmd.append("--gradient_checkpointing")
if QUANTIZE_4BIT:
    # Flag name varies by diffusers version/script; if this errors, check the --help output above and edit this.
    cmd.append("--quantize=nf4")

print("running:", " ".join(cmd))
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed_min = (time.time() - t0) / 60
print(f"\n--- training process exited with code {proc.returncode} after {elapsed_min:.1f} min ---")
if proc.returncode != 0:
    raise SystemExit(
        f"Training script exited with a non-zero code ({proc.returncode}). Scroll up to the actual error - a "
        f"common first failure on an unfamiliar script is a renamed/removed CLI flag; compare against the "
        f"--help output above."
    )

with open(OUTPUT_DIR / "training_config.json", "w") as f:
    json.dump(dict(base_model=BASE_MODEL_ID, trigger=TRIGGER, steps=MAX_TRAIN_STEPS, rank=LORA_RANK,
                   resolution=RESOLUTION, batch_size=BATCH_SIZE, lr=LEARNING_RATE, images=int(len(df)),
                   suppliers=sorted(df["supplier"].unique().tolist()), elapsed_minutes=round(elapsed_min, 1),
                   train_script=TRAIN_SCRIPT), f, indent=2)
print("done - LoRA weights are in", run_output_dir)


## 9. Save for download

In [ ]:
zip_path = shutil.make_archive(str(OUTPUT_DIR / "curalina_sd35_lora"), "zip", str(run_output_dir))
print("zipped to", zip_path)


## 10. Use it: plain model vs. model + your LoRA

Same mechanism as every other notebook here: `pipe.load_lora_weights(...)` is a stable, standard `diffusers` API regardless of the underlying architecture.

In [ ]:
from diffusers import StableDiffusion3Pipeline
import torch as _torch

compare_prompt = f"{TRIGGER}, a sofa and a coffee table in a bright living room, interior design photo"

pipe_kwargs = dict(torch_dtype=_torch.bfloat16)
if QUANTIZE_4BIT:
    from transformers import BitsAndBytesConfig
    pipe_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")

pipe = StableDiffusion3Pipeline.from_pretrained(BASE_MODEL_ID, **pipe_kwargs).to(device)

gen = _torch.Generator(device=device).manual_seed(SEED)
base_image = pipe(compare_prompt, num_inference_steps=28, generator=gen).images[0]

pipe.load_lora_weights(str(run_output_dir))
gen = _torch.Generator(device=device).manual_seed(SEED)
lora_image = pipe(compare_prompt, num_inference_steps=28, generator=gen).images[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(base_image); axes[0].axis("off"); axes[0].set_title("Base Stable Diffusion 3.5 Medium")
axes[1].imshow(lora_image); axes[1].axis("off"); axes[1].set_title("Base + your LoRA")
fig.suptitle(compare_prompt)
fig.tight_layout(); plt.show()


## Using the result later

Same pattern as the other notebooks: a `SD35_LORA_PATH`-style env var analogous to `sd15`'s
`SD15_LORA_PATH` (see `ai_services/renderers/sd15/src/curalina_sd15/pipeline.py` for the pattern a new
`sd35` renderer sidecar would follow — none exists yet in this repo).
